In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[2]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

/Users/rawls/quant-lab


In [2]:
import pandas as pd
import numpy as np

from src.data.market_configs import MARKET_CONFIGS
from src.data.loader import download_market_data
from src.pipelines.strategy_returns import build_strategy_return_stack

from src.utils.metrics import (
    sharpe_ratio,
    max_drawdown,
    annualized_return,
)

from src.analysis.turnover import (
    compute_turnover,
    rolling_turnover,
    summarize_turnover,
    build_turnover_report,
)

In [3]:
#Deployment candidates

market_specs = {
    "India": {
        "config": MARKET_CONFIGS["india"],
        "signals": [
            "mr_ret_10",
            "low_vol_20",
            "mr_lowvol_blend",
        ],
    },
    "Brazil": {
        "config": MARKET_CONFIGS["brazil"],
        "signals": [
            "mom_blend",
            "mr_lowvol_blend",
        ],
    },
    "Japan": {
        "config": MARKET_CONFIGS["japan"],
        "signals": [
            "mr_ret_5",
            "mr_ret_20",
        ],
    },
}

In [4]:
cost_bps_list = [0, 2, 5, 10, 20, 50]

cost_results = []

for market_name, spec in market_specs.items():

    market_data = download_market_data(spec["config"])

    stack = build_strategy_return_stack(
        market_data,
        signal_names=spec["signals"],
        target_vol=0.10,
    )

    returns = stack["dd_vol_ret"]

    weights = (
        stack["panel"]
        .pivot(index="Date", columns="ticker", values="weight")
        .fillna(0)
    )

    turnover = compute_turnover(weights)

    for cost_bps in cost_bps_list:

        cost_rate = cost_bps / 10_000

        net_returns = returns - (turnover.reindex(returns.index).fillna(0) * cost_rate)

        equity = (1 + net_returns).cumprod()
        mdd = max_drawdown(equity)
        cagr = annualized_return(net_returns)

        cost_results.append({
            "Market": market_name,
            "Cost bps": cost_bps,
            "Sharpe": sharpe_ratio(net_returns),
            "MDD": mdd,
            "CAGR": cagr,
            "Calmar": cagr / abs(mdd),
            "Mean Turnover": turnover.mean(),
        })

cost_stress_df = pd.DataFrame(cost_results).round(3)

cost_stress_df

,Market,Cost bps,Sharpe,MDD,CAGR,Calmar,Mean Turnover
0,India,0,2.118,-0.220,0.235,1.071,0.285
1,India,2,1.978,-0.238,0.218,0.914,0.285
2,India,5,1.767,-0.266,0.192,0.720,0.285
3,India,10,1.415,-0.310,0.150,0.482,0.285
4,India,20,0.712,-0.391,0.070,0.179,0.285
5,India,50,-1.394,-0.925,-0.138,-0.149,0.285
6,Brazil,0,1.651,-0.259,0.159,0.615,0.512
7,Brazil,2,1.371,-0.288,0.130,0.451,0.512
8,Brazil,5,0.951,-0.347,0.087,0.251,0.512
9,Brazil,10,0.252,-0.467,0.019,0.041,0.512


In [5]:
turnover.describe()

count    4060.000000
mean        0.325734
std         0.135425
min         0.000000
25%         0.222222
50%         0.333333
75%         0.444444
max         0.857143
dtype: float64

In [6]:
turnover.value_counts().head(20)

0.333333    560
0.222222    440
0.444444    439
0.222222    261
0.285714    210
0.111111    191
0.375000    164
0.142857    144
0.555556    140
0.250000    139
0.500000    123
0.333333    118
0.428571    116
0.250000     90
0.333333     84
0.375000     69
0.444444     58
0.111111     57
0.300000     56
0.000000     49
Name: count, dtype: int64

Note: The current implementation is a daily rebalanced cross-sectional strategy that predicts overlapping 5-day forward returns (fwd_ret_5). Transaction costs are therefore applied daily based on changes in target portfolio weights. This should not be interpreted as a strategy that holds positions unchanged for five trading days. A future robustness test should evaluate a true 5-day rebalance schedule to quantify the trade-off between turnover reduction and alpha retention.

In [7]:
cost_stress_df.to_csv(
    "../results/transaction_cost_stress.csv",
    index=False,
)